# 01 环境配置体检

**测什么**: `.env` 的位置与内容、pydantic-settings 与 `os.getenv` 两条读取路径是否一致、
必填凭据是否就位、数据文件与本地模型缓存是否齐备、日志目录是否可写。

**判定**: `ALL PASSED` 表示配置层没有硬缺口; `HAS FAILURES` 表示有项会直接导致功能不可用。
凭据类取值一律脱敏显示。

**前置**: 无。这是其余各册的前置体检。


In [ ]:
import asyncio, os, sys
from pathlib import Path

for cand in (Path.cwd(), *Path.cwd().parents):
    if (cand / "nbkit.py").is_file():
        NB_DIR = cand
        break
    if (cand / "tests_ipynb" / "nbkit.py").is_file():
        NB_DIR = cand / "tests_ipynb"
        break
else:
    raise RuntimeError("未找到 nbkit.py")

sys.path.insert(0, str(NB_DIR))

from nbkit import Checks, bootstrap

ROOT = bootstrap()
checks = Checks("01 环境配置体检")

print("解释器  :", sys.executable)
print("仓库根  :", ROOT)
print("HF_HOME :", os.getenv("HF_HOME", "(未设置)"))


## 1. .env 文件位置与内容

In [ ]:
from nbkit import env_path, parse_env, mask

path = env_path(ROOT)
checks.expect(
    path.is_file(),
    ".env 存在于包内约定路径",
    ok_detail=str(path.relative_to(ROOT)),
    fail_detail="缺失 → 执行 cp lawApp_LangGraph/.env.example lawApp_LangGraph/.env",
)
raw = parse_env(ROOT)
print(f"文件内键值对: {len(raw)} 项")
for k in sorted(raw):
    print(f"  {k:24s} = {mask(raw[k])}")

## 2. settings 是否读到文件里的值

In [ ]:
from lawApp_LangGraph.config import settings as s

pairs = [
    ("DEEPSEEK_API_KEY", raw.get("DEEPSEEK_API_KEY", ""), s.deepseek_api_key or ""),
    ("DEEPSEEK_BASE_URL", raw.get("DEEPSEEK_BASE_URL", ""), s.deepseek_base_url),
    ("CHECKPOINT_BACKEND", raw.get("CHECKPOINT_BACKEND", ""), s.checkpoint_backend),
    ("RETRIEVER_BACKEND", raw.get("RETRIEVER_BACKEND", ""), None),
    ("DB_HOST", raw.get("DB_HOST", ""), s.db_host),
    ("DB_PORT", raw.get("DB_PORT", ""), str(s.db_port)),
    ("DB_USER", raw.get("DB_USER", ""), s.db_user),
    ("DB_NAME", raw.get("DB_NAME", ""), s.db_name),
    ("MCP_SERVER_URL", raw.get("MCP_SERVER_URL", ""), s.mcp_server_url),
    ("MAX_ROUNDS", raw.get("MAX_ROUNDS", ""), str(s.max_rounds)),
    ("LOG_DIR", raw.get("LOG_DIR", ""), s.log_dir),
]
for name, fv, sv in pairs:
    if sv is None:
        checks.skip(f"settings 读取: {name}", "该键不由 config.py 收拢(见下一节)")
        continue
    secret = "KEY" in name or "PASSWORD" in name
    checks.expect(
        fv == sv,
        f"settings 读取一致: {name}",
        ok_detail=mask(sv) if secret else str(sv),
        fail_detail=(
            f"文件={mask(fv)} 但 settings={mask(sv)}"
            if secret
            else f"文件={mask(fv)} 但 settings={mask(sv)}"
        ),
    )

## 3. `os.getenv` 读取路径(仅 `api.py` 入口会 `load_dotenv`)

`RETRIEVER_BACKEND` / `LEGAL_ANALYSIS_ROLE` / `PDF_OUTPUT_DIR` 由代码直接 `os.getenv` 读取,
不走 pydantic-settings。只有调过 `load_dotenv()` 的进程(`api.py`)能看到文件里的值;
直接跑脚本或 MCP server 时取代码默认值。

In [ ]:
import os
from dotenv import load_dotenv

env_only = ("RETRIEVER_BACKEND", "LEGAL_ANALYSIS_ROLE", "PDF_OUTPUT_DIR")
filed = {k: raw.get(k) for k in env_only}
before = {k: os.getenv(k) for k in env_only}
load_dotenv(dotenv_path=path, override=False)
after = {k: os.getenv(k) for k in env_only}

print("文件里写的            :", filed)
print("load_dotenv 之前      :", before)
print("load_dotenv 之后      :", after)
print()
for k in env_only:
    if filed.get(k):
        checks.expect(
            after.get(k) == filed[k],
            f"load_dotenv 后可读: {k}",
            ok_detail=f"={after.get(k)}",
            fail_detail=f"文件写了一致值但读不到: 文件={filed[k]} 实际={after.get(k)}",
        )
    else:
        checks.skip(f"文件未写: {k}", f"走代码默认值; 当前 os.getenv={before.get(k)}")

## 4. 必填凭据

In [ ]:
key = s.deepseek_api_key or ""
checks.expect(
    bool(key.strip()),
    "DEEPSEEK_API_KEY 已填",
    ok_detail=mask(key),
    fail_detail="为空 → planner/executor/finalize 全部失败, 图停在 degrade 询问",
)
checks.expect(bool(s.deepseek_base_url.strip()), "DEEPSEEK_BASE_URL 非空", s.deepseek_base_url)
checks.expect(
    s.deepseek_pro_model == "deepseek-reasoner" and s.deepseek_flash_model == "deepseek-chat",
    "双 LLM 模型名与设计一致",
    f"{s.deepseek_pro_model} / {s.deepseek_flash_model}",
)

backend = (raw.get("RETRIEVER_BACKEND") or "pgvector").strip().lower()
serp = (os.getenv("SERPAPI_API_KEY") or raw.get("SERPAPI_API_KEY") or "").strip()
pine = (s.pinecone_api_key or "").strip()
checks.skip("SERPAPI_API_KEY", mask(serp) if serp else "未填 → 联网兜底(get_google_search)不可用")
if backend == "pinecone":
    checks.expect(bool(pine), "PINECONE_API_KEY 已填(pinecone 后端)", mask(pine), "后端=pinecone 却无凭据")
else:
    checks.skip("PINECONE_API_KEY", f"当前 RETRIEVER_BACKEND={backend}, 走本地 pgvector 无需凭据")

## 5. 数据文件

In [ ]:
bm25 = Path(s.bm25_path) if s.bm25_path else None
checks.expect(
    bool(bm25 and bm25.is_file()),
    "BM25 参数文件存在",
    ok_detail=str(bm25),
    fail_detail=f"缺失: {bm25}",
)

docs = Path(s.documents_dir)
checks.expect(bool(docs.is_dir()), "DOCUMENTS_DIR 存在", str(docs), f"缺失: {docs}")
if docs.is_dir():
    laws = sorted(docs.glob("LawDocument/*.txt"))
    cases = sorted(docs.glob("MarkDownFiles/*.md"))
    checks.expect(len(laws) > 0, f"法条语料 {len(laws)} 个", ok_detail=f"{len(laws)} 个 .txt")
    checks.expect(len(cases) > 0, f"案例语料 {len(cases)} 个", ok_detail=f"{len(cases)} 个 .md")

## 6. 本地模型缓存

In [ ]:
from nbkit import hf_hub_dir, hf_model_cached

hub = hf_hub_dir()
checks.expect(hub.is_dir(), "HF 缓存目录存在", str(hub), f"不存在: {hub} → 加载模型需联网")
for repo in (s.memory_embed_model, s.rerank_model):
    cached = hf_model_cached(repo)
    (checks.ok if cached else checks.fail)(
        f"模型已缓存: {repo}",
        "离线可加载" if cached else "未缓存 → 首次使用会联网下载, 离线环境将失败",
    )

## 7. 日志目录可写

In [ ]:
import tempfile

logdir = (ROOT / s.log_dir).resolve() if not Path(s.log_dir).is_absolute() else Path(s.log_dir)
if logdir.is_dir():
    target = logdir
    note = str(logdir)
else:
    target = Path(tempfile.mkdtemp(prefix="lawapp_log_probe_"))
    note = f"{logdir} 尚不存在(相对路径按进程 CWD 解析); 改在临时目录试探: {target}"

try:
    probe = target / ".write_probe"
    probe.write_text("ok", encoding="utf-8")
    probe.unlink()
    checks.ok("日志目录可写", note)
except Exception as e:
    checks.fail("日志目录可写", f"{type(e).__name__}: {str(e)[:120]}")

## 汇总

In [ ]:
print(checks.report())